# Advanced Naive Bayes Exploration: Audio vs. Lyrics

This notebook conducts a comprehensive comparison of different Naive Bayes variants for predicting song popularity. We explore both **Audio Features** and **Lyrics** to see which provides better predictive power.

### Objectives:
1. **Data Integration**: Download and merge multiple datasets (Spotify audio features and lyrics).
2. **Audio-Based Models**: Compare Gaussian and Categorical Naive Bayes on continuous and discretized audio data.
3. **Lyrics-Based Model**: Use Multinomial Naive Bayes on TF-IDF transformed lyrics.
4. **Performance Optimization**: Apply SMOTE to address class imbalance in the popularity dataset.

## 1) Environment Setup and Data Download

We use the `kagglehub` API to pull two different datasets: 
- A popularity classification dataset for audio features.
- A lyrics dataset for textual analysis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

# Machine Learning and Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, CategoricalNB, MultinomialNB
from sklearn.preprocessing import KBinsDiscretizer, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# Natural Language Processing (NLP) tools
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download datasets
audio_path = kagglehub.dataset_download("lynnxxx/spotify-tracks-popularity-classification")
lyrics_path = kagglehub.dataset_download("evabot/spotify-lyrics-dataset")

print("Audio Dataset Path:", audio_path)
print("Lyrics Dataset Path:", lyrics_path)

## 2) Data Loading and Preparation

We load both datasets and prepare a common identifier (`track_id`) to merge them later. We also define our target variable: a song is a "hit" if its popularity is 50 or greater.

In [ ]:
# Load Audio Features Dataset
csv_files_audio = [f for f in os.listdir(audio_path) if f.endswith('.csv')]
df_audio = pd.read_csv(os.path.join(audio_path, csv_files_audio[0]))

# Load Lyrics Dataset (using the robust logic from previous work)
csv_files_lyrics = [f for f in os.listdir(lyrics_path) if f.endswith('.csv')]
lyrics_file = next((f for f in csv_files_lyrics if 'lyrics' in f.lower()), csv_files_lyrics[0])
df_lyrics = pd.read_csv(os.path.join(lyrics_path, lyrics_file), sep=';', decimal=',', on_bad_lines='skip')
if 'lyrics' not in df_lyrics.columns and 'Lyrics' in df_lyrics.columns:
    df_lyrics = df_lyrics.rename(columns={'Lyrics': 'lyrics'})
df_lyrics = df_lyrics.dropna(subset=['lyrics'])

# Define 'is_hit' (Target Variable)
df_audio['is_hit'] = (df_audio['popularity'] >= 50).astype(int)

print(f"Audio Data Shape: {df_audio.shape}")
print(f"Lyrics Data Shape: {df_lyrics.shape}")
print("\nHit distribution in Audio Data:")
print(df_audio['is_hit'].value_counts())

## 3) Gaussian Naive Bayes (Continuous Audio Features)

Gaussian NB assumes that continuous features follow a normal distribution. We use standard audio features like danceability, energy, and loudness to predict if a song is a hit.

In [ ]:
features_audio = ['acousticness', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'speechiness', 'instrumentalness']
X_audio = df_audio[features_audio]
y_audio = df_audio['is_hit']

# Train-Test Split
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_audio, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

# Train Gaussian NB
model_gnb = GaussianNB()
model_gnb.fit(X_train_g, y_train_g)

# Evaluate
y_pred_gnb = model_gnb.predict(X_test_g)
print("Gaussian NB (Audio Features) Performance:")
print(f"Accuracy: {accuracy_score(y_test_g, y_pred_gnb):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_g, y_pred_gnb))

## 4) Categorical Naive Bayes (Discretized Audio Features)

Categorical NB requires discrete inputs. We use `KBinsDiscretizer` to convert our continuous audio measurements into 3 categories (low, medium, high) based on uniform width to avoid issues with skewed data like instrumentalness.

In [ ]:
# Discretize features into 3 bins using uniform strategy
discretizer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', subsample=None, random_state=42)
X_audio_binned = discretizer.fit_transform(X_audio)

# Train-Test Split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_audio_binned, y_audio, test_size=0.3, random_state=42, stratify=y_audio
)

# Train Categorical NB
model_cnb = CategoricalNB()
model_cnb.fit(X_train_c, y_train_c)

# Evaluate
y_pred_cnb = model_cnb.predict(X_test_c)
print("Categorical NB (Discretized Audio) Performance:")
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_cnb):.4f}")

## 5) Lyrics-Based Popularity Prediction

Now we switch to lyrics. First, we merge the popularity info from the audio dataset into the lyrics dataset using Spotify Track IDs. Then we clean the text and apply Multinomial Naive Bayes.

In [ ]:
# Prepare IDs for merging
df_lyrics['clean_id'] = df_lyrics['song_id'].apply(lambda x: x.split(':')[-1] if isinstance(x, str) else np.nan)

# Merge Popularity into Lyrics
df_lyrics_pop = pd.merge(
    df_lyrics,
    df_audio[['id', 'is_hit']],
    left_on='clean_id',
    right_on='id',
    how='inner'
)

print(f"Songs with both Lyrics and Popularity Info: {len(df_lyrics_pop)}")

# Basic Cleaning and NLP Setup
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_lyrics(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    return ' '.join([lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1])

# Apply cleaning
df_lyrics_pop['cleaned'] = df_lyrics_pop['lyrics'].apply(clean_lyrics)

# Split first to avoid leakage
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    df_lyrics_pop['cleaned'], df_lyrics_pop['is_hit'], test_size=0.2, random_state=42, stratify=df_lyrics_pop['is_hit']
)

# TF-IDF Vectorization (fitted ONLY on train)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train_l)
X_test_tfidf = vectorizer.transform(X_test_l)

print("Lyrics feature matrix shape:", X_train_tfidf.shape)

## 6) Handling Imbalance with SMOTE

Lyrics datasets are often imbalanced (many more non-hits than hits). We use **SMOTE** to create synthetic examples of hit songs, helping the Multinomial NB model learn what a "hit" sounds like.

In [ ]:
# Apply SMOTE to balance the training set
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_tfidf, y_train_l)

# Train Multinomial NB on balanced data
model_mnb = MultinomialNB()
model_mnb.fit(X_train_balanced, y_train_balanced)

# Evaluate
y_pred_mnb = model_mnb.predict(X_test_tfidf)
print("Multinomial NB (Balanced Lyrics) Performance:")
print(f"Accuracy: {accuracy_score(y_test_l, y_pred_mnb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_l, y_pred_mnb))

## 7) Results Visualization

Finally, we visualize the confusion matrix for our lyrics-based model to see how well it distinguishes between hits and non-hits.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test_l, y_pred_mnb), annot=True, fmt='d', cmap='Reds',
            xticklabels=['Predicted Non-Hit', 'Predicted Hit'],
            yticklabels=['Actual Non-Hit', 'Actual Hit'])
plt.title('Confusion Matrix: Lyrics-Based Hit Prediction (with SMOTE)')
plt.show()

## Summary

This exploration demonstrates that while audio features provide a solid baseline, song lyrics offer deep semantic insights into song popularity. By combining advanced techniques like **TF-IDF vectorization**, **discretization**, and **SMOTE oversampling**, we can build more robust models that better understand the characteristics of hit songs.